# Lightweight File Tree (`solveitFileTree`)

> Sidebar file browser (`Mod+b`) with keyboard navigation, context menu, drag-and-drop uploads, and ZIP-backed folder operations.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
UPLOAD_ICON = svg_icon(['M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4', ft('polyline', points='17 8 12 3 7 8'), ft('line', x1='12', y1='3', x2='12', y2='15')], size=14, sw=2)
COLLAPSE_ICON = svg_icon([ft('polyline', points='7 20 12 15 17 20'), ft('polyline', points='7 4 12 9 17 4'), ft('line', x1='4', y1='12', x2='20', y2='12')], size=14, sw=2)

TEMPLATES = dict(
    ft_upload_icon=UPLOAD_ICON,
    ft_collapse_icon=COLLAPSE_ICON,
    ft_temp_editor=Div(
        Div(Strong(), Button('×', type='button', aria_label='Close editor'), klass='solveit-temp-editor-bar'),
        Iframe(title='Temporary file editor'),
        id='solveit-temp-editor'),
    ft_create_form=Form(
        Input(type='text', autocomplete='off', spellcheck='false', enterkeyhint='done', aria_describedby='solveit-tree-create-error'),
        Div(id='solveit-tree-create-error', role='alert'),
        klass='solveit-tree-create'),
    ft_upload_pop=Div(
        Div(Span(klass='solveit-tree-upload-dest'), Button('×', type='button', klass='solveit-tree-upload-x', aria_label='Close'), klass='solveit-tree-upload-head'),
        Div('Drop files or folders here', klass='solveit-tree-upload-drop'),
        Div(Button('Files…', type='button', klass='solveit-tree-upload-pick-files'), Button('Folder…', type='button', klass='solveit-tree-upload-pick-dir'), klass='solveit-tree-upload-actions'),
        Input(type='file', klass='solveit-tree-upload-input-files', multiple=True, hidden=True),
        Input(type='file', klass='solveit-tree-upload-input-dir', webkitdirectory=True, multiple=True, hidden=True),
        klass='solveit-tree-upload-pop', role='dialog'),
    ft_panel=Aside(
        Div(Span('solveit', klass='solveit-tree-appname'), klass='solveit-tree-brand'),
        Header(
            Strong('Files'),
            ft('kbd', '⌘B', klass='solveit-tree-shortcut', data_hotkey='file-tree-toggle'),
            Button('+', klass='solveit-tree-new', type='button', aria_label='New dialog, file, or folder', title='New dialog, file, or folder (name/ for folder)'),
            Button(UPLOAD_ICON, klass='solveit-tree-upload', type='button', aria_label='Upload files or folders', title='Upload files or folders'),
            Button(COLLAPSE_ICON, klass='solveit-tree-collapse', type='button', aria_label='Collapse all folders', title='Collapse all folders'),
            Button('.*', klass='solveit-tree-dots', type='button', aria_pressed='true'),
            Button('↻', klass='solveit-tree-refresh', type='button', aria_label='Refresh files', title='Refresh files'),
            Button('⌂', klass='solveit-tree-home', type='button', aria_label='Dialogs home', title='Dialogs home'),
        ),
        Input(klass='solveit-tree-filter', type='search', placeholder='Filter loaded files…', autocomplete='off'),
        Div(klass='solveit-tree-status', role='status'),
        Div(
            Span('~/', klass='solveit-tree-rootbar-path'),
            Span('root · drop / right-click', klass='solveit-tree-rootbar-hint'),
            Button(UPLOAD_ICON, klass='solveit-tree-row-upload', type='button', aria_label='Upload into root', title='Upload into root'),
            Button('⋯', klass='solveit-tree-row-more', type='button', aria_label='Actions for root', title='New dialog, file, folder…'),
            klass='solveit-tree-rootbar', title='Right-click to create/paste/upload in root; drop here to move or upload to root'),
        Div(id='solveit-file-tree', role='tree'),
        Div(klass='solveit-tree-resizer', aria_hidden='true', title='Drag to resize · Double-click to reset'),
        id='solveit-file-tree-panel', role='complementary', aria_label='Files'),
    ft_menu=Div(klass='solveit-tree-menu', role='menu'),
    ft_workspace=Div(klass='solveit-tree-workspace'),
)

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Lightweight file tree ─────────────────────────────────────────── */
(() => {
  window.solveitFileTree?.disable?.();
  const {mountTpl, qs, on, store, patch} = window.__craftCMCore;
  const QUERCUS_JS = '/static/.craft-assets/quercus-treeview.js', QUERCUS_CSS = '/static/.craft-assets/quercus-treeview.css';
  const DOTS_KEY = 'solveit-file-tree:hide-dots';
  const CACHE_KEY = 'solveit-file-tree:v4', CLIP_KEY = 'solveit-file-tree:clipboard', CACHE_MAX_AGE = 180000;
  let panel, tree, loaded = false, filterText = '', restoring = false, previousFocus;
  let refreshTimer, busy = false, menu = null, uploadPop = null, lastTreeDir = null;
  const UPLOAD_SVG = window.__craftTpl?.ft_upload_icon || '', COLLAPSE_SVG = window.__craftTpl?.ft_collapse_icon || '';
  let workspaceRow = null, dlgNameNode = null, dlgNameOrigParent = null, dlgNameOrigNext = null;
  let draggedNode = null, dropHighlightEl = null, hoverExpandTimer = null, hoverExpandPath = null;

  let hideDots = store.get(DOTS_KEY, '1') === '1';

  const OPEN_KEY = 'solveit-file-tree:open';
  const saveOpen = open => store.set(OPEN_KEY, open ? '1' : '0', sessionStorage);
  window.__hotkeyRegister?.({id: 'file-tree-toggle', label: 'Toggle file tree', group: 'File tree', combo: 'Mod+B', handler: () => { toggle(); return true; }});

  const rowsByPath = new Map(), fetchedAtByPath = new Map(), expandedPaths = new Set(), loadingPaths = new Set();
  const cached = store.json(CACHE_KEY, null, sessionStorage);
  if (cached) {
    cached.rows?.forEach(([path, rows]) => rowsByPath.set(path, rows));
    cached.fetchedAt?.forEach(([path, time]) => fetchedAtByPath.set(path, time));
    cached.expanded?.forEach(path => expandedPaths.add(path));
  }
  const saveCache = () => store.setJson(CACHE_KEY, {rows: [...rowsByPath], fetchedAt: [...fetchedAtByPath], expanded: [...expandedPaths]}, sessionStorage);

  /* ── Path & clipboard helpers ──────────────────────────────────────── */
  const normPath = p => {
    const s = String(p || '').trim().replace(/^\/+/, '').replace(/^\.\//, '').replace(/\/+$/, '');
    return s || '.';
  };
  const parentOf = p => {
    const n = normPath(p);
    if (n === '.') return '.';
    const i = n.lastIndexOf('/');
    return i > 0 ? n.slice(0, i) : '.';
  };
  const baseOf = p => {
    const n = normPath(p);
    if (n === '.') return '.';
    const i = n.lastIndexOf('/');
    return i >= 0 ? n.slice(i + 1) : n;
  };
  const joinPath = (dir, name) => (!dir || dir === '.' ? name : `${dir}/${name}`);
  const staticUrl = p => '/static/' + normPath(p).split('/').map(encodeURIComponent).join('/') + '?download=true';
  const ancestorsOf = dir => {
    const n = normPath(dir);
    if (n === '.') return ['.'];
    const parts = n.split('/'), out = ['.'];
    for (let i = 0; i < parts.length; i++) out.push(parts.slice(0, i + 1).join('/'));
    return out;
  };
  function getCurrentContext() {
    const params = new URLSearchParams(location.search);
    if (location.pathname.startsWith('/dialog_')) {
      const raw = normPath(params.get('name') || '');
      if (raw !== '.') {
        const dlgPath = raw.endsWith('.ipynb') ? raw : `${raw}.ipynb`;
        return {dir: parentOf(dlgPath), dlgPath};
      }
    }
    if (location.pathname === '/') return {dir: normPath(params.get('at') || '.'), dlgPath: null};
    return {dir: '.', dlgPath: null};
  }
  const getClipboard = () => store.json(CLIP_KEY, null, sessionStorage);
  const setClipboard = item => item ? store.setJson(CLIP_KEY, item, sessionStorage) : store.del(CLIP_KEY, sessionStorage);
  function setStatus(msg) {
    const el = qs('.solveit-tree-status', panel);
    if (el) el.textContent = msg || '';
  }

  /* ── Minimal ZIP builders (for mkdir & renamed recursive folder copy) ── */
  function makeEmptyDirZip(dirName) {
    const nameBytes = new TextEncoder().encode(dirName.replace(/\/+$/, '') + '/');
    const n = nameBytes.length;
    const lfh = new Uint8Array(30 + n), lv = new DataView(lfh.buffer);
    lv.setUint32(0, 0x04034b50, true); lv.setUint16(4, 20, true); lv.setUint16(26, n, true);
    lfh.set(nameBytes, 30);
    const cdh = new Uint8Array(46 + n), cv = new DataView(cdh.buffer);
    cv.setUint32(0, 0x02014b50, true); cv.setUint16(4, 0x0314, true); cv.setUint16(6, 20, true);
    cv.setUint16(28, n, true); cv.setUint32(38, 0x41ed0010, true); cv.setUint32(42, 0, true);
    cdh.set(nameBytes, 46);
    const eocd = new Uint8Array(22), ev = new DataView(eocd.buffer);
    ev.setUint32(0, 0x06054b50, true); ev.setUint16(8, 1, true); ev.setUint16(10, 1, true);
    ev.setUint32(12, cdh.length, true); ev.setUint32(16, lfh.length, true);
    return new Blob([lfh, cdh, eocd], {type: 'application/zip'});
  }
  function renameZipTopDir(bytes, oldTop, newTop) {
    const view = new DataView(bytes.buffer, bytes.byteOffset, bytes.byteLength);
    const dec = new TextDecoder(), enc = new TextEncoder();
    let eocd = bytes.length - 22;
    while (eocd >= 0 && view.getUint32(eocd, true) !== 0x06054b50) eocd--;
    if (eocd < 0) throw new Error('Invalid zip archive');
    const cdCount = view.getUint16(eocd + 10, true);
    if (cdCount === 0xffff) throw new Error('ZIP64 archives are not supported');
    const cdOff = view.getUint32(eocd + 16, true);
    const entries = [];
    let p = cdOff;
    for (let i = 0; i < cdCount; i++) {
      const nLen = view.getUint16(p + 28, true);
      const exLen = view.getUint16(p + 30, true);
      const cmLen = view.getUint16(p + 32, true);
      const lfhOff = view.getUint32(p + 42, true);
      const name = dec.decode(bytes.subarray(p + 46, p + 46 + nLen));
      entries.push({cdOff: p, cdLen: 46 + nLen + exLen + cmLen, nLen, lfhOff, name});
      p += 46 + nLen + exLen + cmLen;
    }
    const byLfh = [...entries].sort((a, b) => a.lfhOff - b.lfhOff);
    const prefix = oldTop.replace(/\/+$/, '') + '/';
    const newPrefix = newTop.replace(/\/+$/, '') + '/';
    const localParts = [];
    let curOff = 0;
    for (let i = 0; i < byLfh.length; i++) {
      const e = byLfh[i];
      const nextOff = i + 1 < byLfh.length ? byLfh[i + 1].lfhOff : cdOff;
      const origLfhNameLen = view.getUint16(e.lfhOff + 26, true);
      const renamed = e.name.startsWith(prefix) ? newPrefix + e.name.slice(prefix.length) : newPrefix + e.name;
      e.newNameBytes = enc.encode(renamed);
      e.newLfhOff = curOff;
      const head = new Uint8Array(bytes.subarray(e.lfhOff, e.lfhOff + 30));
      new DataView(head.buffer).setUint16(26, e.newNameBytes.length, true);
      const rest = bytes.subarray(e.lfhOff + 30 + origLfhNameLen, nextOff);
      localParts.push(head, e.newNameBytes, rest);
      curOff += head.length + e.newNameBytes.length + rest.length;
    }
    const newCdOff = curOff, cdParts = [];
    let newCdSize = 0;
    for (const e of entries) {
      const head = new Uint8Array(bytes.subarray(e.cdOff, e.cdOff + 46));
      const hv = new DataView(head.buffer);
      hv.setUint16(28, e.newNameBytes.length, true);
      hv.setUint32(42, e.newLfhOff, true);
      const rest = bytes.subarray(e.cdOff + 46 + e.nLen, e.cdOff + e.cdLen);
      cdParts.push(head, e.newNameBytes, rest);
      newCdSize += head.length + e.newNameBytes.length + rest.length;
    }
    const newEocd = new Uint8Array(bytes.subarray(eocd));
    const ev = new DataView(newEocd.buffer);
    ev.setUint32(12, newCdSize, true);
    ev.setUint32(16, newCdOff, true);
    return new Blob([...localParts, ...cdParts, newEocd], {type: 'application/zip'});
  }

  /* ── Backend requests ──────────────────────────────────────────────── */
  function loadAssets() {
    if (loaded || typeof window.Treeview === 'function') { loaded = true; return Promise.resolve(); }
    if (!document.querySelector('link[data-solveit-tree="1"]')) {
      const css = document.createElement('link');
      css.rel = 'stylesheet'; css.href = QUERCUS_CSS; css.dataset.solveitTree = '1'; document.head.append(css);
    }
    return new Promise((resolve, reject) => {
      const script = document.createElement('script');
      script.src = QUERCUS_JS; script.dataset.solveitTree = '1';
      script.onload = () => { loaded = true; resolve(); };
      script.onerror = () => reject(new Error('Could not load Quercus assets'));
      document.head.append(script);
    });
  }

  async function getRows(dirPath) {
    const cleanDir = normPath(dirPath);
    const response = await fetch(`/?at=${encodeURIComponent(cleanDir)}`, {method: 'POST', credentials: 'same-origin', headers: {'HX-Request': 'true'}});
    if (!response.ok) throw new Error(`File listing failed: ${response.status}`);
    const doc = new DOMParser().parseFromString(await response.text(), 'text/html');
    return [...doc.querySelectorAll('.tbl-row')].map(row => {
      const link = row.querySelector('a.file-link');
      if (!link) return null;
      const href = link.getAttribute('href'), hxPost = link.getAttribute('hx-post'), isDir = !!hxPost;
      const kind = isDir ? 'directory' : href?.startsWith('/dialog_?') ? 'dialog' : 'file';
      const label = link.textContent.trim();
      let relPath = isDir
        ? new URL(hxPost, location.href).searchParams.get('at')
        : (link.getAttribute('title') || null);
      if (!relPath && href && href !== '#') {
        const u = new URL(href, location.href);
        relPath = u.searchParams.get('fp') || u.searchParams.get('name');
      }
      relPath = normPath(relPath || joinPath(cleanDir, kind === 'dialog' && !label.endsWith('.ipynb') ? `${label}.ipynb` : label));
      const filename = baseOf(relPath);
      return {
        id: relPath,
        name: label,
        filename,
        kind,
        href: href && href !== '#' ? href : null,
        path: relPath,
        parentPath: parentOf(relPath),
        delUrl: row.querySelector('a.file-del')?.getAttribute('hx-post') || null,
        dlUrl: row.querySelector('a[download]')?.getAttribute('href') || null,
      };
    }).filter(Boolean);
  }

  async function uploadBlob(folder, filename, blob, extract = false) {
    const fd = new FormData();
    fd.append('files', new File([blob], filename, {type: blob.type || 'application/octet-stream'}));
    if (extract) fd.append('extract_archives', 'on');
    const r = await fetch(`/upload_file_?folder=${encodeURIComponent(normPath(folder))}`, {
      method: 'POST', credentials: 'same-origin', headers: {'HX-Request': 'true'}, body: fd,
    });
    if (!r.ok) throw new Error(`Upload failed: HTTP ${r.status} ${(await r.text()).trim()}`);
  }

  async function downloadBytes(node) {
    const url = node.dlUrl || staticUrl(node.path);
    const r = await fetch(new URL(url, location.href), {credentials: 'same-origin'});
    if (!r.ok) throw new Error(`Download failed: HTTP ${r.status}`);
    return new Uint8Array(await r.arrayBuffer());
  }

  async function deleteRecursive(node) {
    if (node.kind === 'directory') {
      const children = await getRows(node.path);
      for (const child of children) await deleteRecursive(child);
    }
    const url = node.delUrl || (node.kind === 'dialog'
      ? `/rm_dialog_?name=${encodeURIComponent(node.path)}`
      : `/delete_file_?filename=${encodeURIComponent(node.path)}`);
    const r = await fetch(new URL(url, location.href), {method: 'POST', credentials: 'same-origin', headers: {'HX-Request': 'true'}});
    if (!r.ok) {
      const detail = (await r.text()).trim();
      throw new Error(detail || `HTTP ${r.status}`);
    }
  }

  function purgeDirCache(dirPath) {
    const prefix = dirPath + '/';
    [...rowsByPath.keys()].filter(p => p === dirPath || p.startsWith(prefix)).forEach(p => {
      rowsByPath.delete(p); fetchedAtByPath.delete(p); expandedPaths.delete(p);
    });
  }

  /* ── Tree rendering & navigation ───────────────────────────────────── */
  const KIND_ORDER = {directory: 0, dialog: 1, file: 2};
  function nodesFor(path, curAncestors) {
    return (rowsByPath.get(path) || [])
      .filter(n => {
        const fname = n.filename || n.name || '';
        if (!hideDots || !fname.startsWith('.')) return true;
        return n.kind === 'directory' && curAncestors.has(n.path);
      })
      .sort((a, b) => {
        const ka = KIND_ORDER[a.kind] ?? 3, kb = KIND_ORDER[b.kind] ?? 3;
        if (ka !== kb) return ka - kb;
        return (a.name || '').localeCompare(b.name || '', undefined, {numeric: true, sensitivity: 'base'});
      })
      .map(node => node.kind !== 'directory' ? node : ({
        ...node,
        children: rowsByPath.has(node.path)
          ? nodesFor(node.path, curAncestors)
          : [{id: `loading:${node.path}`, name: 'Loading…', kind: 'loading', selectable: false}],
      }));
  }

  function findNode(path) {
    if (!panel) return null;
    const esc = typeof CSS !== 'undefined' && CSS.escape ? CSS.escape(path) : path.replace(/["\\]/g, '\\$&');
    const direct = panel.querySelector(`#solveit-file-tree li[data-path="${esc}"]`);
    if (direct) return direct;
    return [...panel.querySelectorAll('#solveit-file-tree li[data-node-data]')]
      .find(li => JSON.parse(li.dataset.nodeData).path === path) || null;
  }

  function focusPath(path) {
    const li = findNode(path);
    const content = li?.querySelector(':scope > .treeview-node-content');
    if (content) {
      content.scrollIntoView({block: 'nearest'});
      content.focus();
      return true;
    }
    return false;
  }

  function renderNode(node, content) {
    const {dir, dlgPath} = getCurrentContext();
    content.classList.add('solveit-tree-' + node.kind);
    content.tabIndex = 0;
    content.setAttribute('role', 'treeitem');
    // Touch: no drag (long-press would start a drag instead of the context menu); rows get a ⋯ menu button below.
    const touch = window.craftMobile?.active();
    if (node.kind !== 'loading' && !touch) content.draggable = true;
    if (node.kind === 'directory') content.setAttribute('aria-expanded', String(expandedPaths.has(node.path)));
    if ((dlgPath && node.path === dlgPath) || (!dlgPath && dir !== '.' && node.path === dir)) {
      content.classList.add('solveit-tree-current');
    }
    if (node.kind !== 'loading') {
      const fname = node.filename || node.name || '';
      const ext = fname.includes('.') ? fname.split('.').pop().toLowerCase() : '';
      const icons = {js: 'file_type_js.svg', css: 'file_type_css.svg', json: 'file_type_json.svg'};
      const icon = node.kind === 'directory'
        ? (expandedPaths.has(node.path) ? 'default_folder_opened.svg' : 'default_folder.svg')
        : node.kind === 'dialog' ? 'file_type_ipynb.svg' : (icons[ext] || 'default_file.svg');
      content.append(window.__craftCMCore.el('img', {cls: 'solveit-tree-icon', src: `/static/.craft-assets/vscode-icons/${icon}`, alt: ''}));
    }
    content.append(window.__craftCMCore.el('span', {cls: 'treeview-node-text'}, node.name));
    if (node.kind === 'directory') {
      const up = window.__craftCMCore.el('button', {
        type: 'button', cls: 'solveit-tree-row-upload', tabIndex: -1,
        title: `Upload into ${node.name}`, 'aria-label': `Upload into ${node.name}`,
      });
      up.innerHTML = UPLOAD_SVG;
      content.append(up);
    }
    if (touch && node.kind !== 'loading') {
      const t = `Actions for ${node.name}`;
      content.append(window.__craftCMCore.el('button', {
        type: 'button', cls: 'solveit-tree-row-more', tabIndex: -1, title: t, 'aria-label': t,
      }, '⋯'));
    }
  }

  function renderTree() {
    if (!panel) return;
    const treeEl = panel.querySelector('#solveit-file-tree');
    const prevScroll = treeEl ? treeEl.scrollTop : 0;
    const focused = createDraft?.input;
    const draftFocus = focused && document.activeElement === focused ? [focused.selectionStart, focused.selectionEnd] : null;
    const {dir} = getCurrentContext();
    const curAncestors = new Set(ancestorsOf(dir));
    const data = nodesFor('.', curAncestors);
    if (!tree) tree = new Treeview({containerId: 'solveit-file-tree', data, initiallyExpanded: false, nodeSelectionEnabled: false, onRenderNode: renderNode});
    else tree.setData(data);
    const nodeIndex = new Map();
    for (const li of panel.querySelectorAll('#solveit-file-tree li[data-node-data]')) {
      try {
        const p = JSON.parse(li.dataset.nodeData).path;
        if (p) { li.dataset.path = p; nodeIndex.set(p, li); }
      } catch {}
    }
    restoring = true;
    [...expandedPaths].sort((a, b) => a.length - b.length).forEach(path => {
      const li = nodeIndex.get(path), expander = li?.querySelector(':scope > .treeview-node-content > .treeview-expander');
      if (expander && !li.classList.contains('expanded')) expander.click();
    });
    restoring = false;
    if (filterText) tree.search(filterText);
    mountCreate();
    if (draftFocus && createDraft?.input.isConnected) {
      const input = createDraft.input;
      input.focus({preventScroll: true});
      input.setSelectionRange(...draftFocus);
    }
    const rootLabel = panel.querySelector('.solveit-tree-rootbar-path');
    if (rootLabel) rootLabel.textContent = dir === '.' ? '~/' : `~/${dir}`;
    if (treeEl && prevScroll) treeEl.scrollTop = prevScroll;
  }

  async function loadFolder(path, force = false) {
    const clean = normPath(path);
    if ((!force && rowsByPath.has(clean)) || loadingPaths.has(clean)) return;
    loadingPaths.add(clean);
    try {
      rowsByPath.set(clean, await getRows(clean));
      fetchedAtByPath.set(clean, Date.now());
      saveCache();
      renderTree();
    } catch (err) { setStatus(`Could not load folder: ${err.message || err}`); }
    finally { loadingPaths.delete(clean); }
  }

  async function refreshDirs(...dirs) {
    const targets = [...new Set(dirs.map(normPath))];
    await Promise.all(targets.map(async p => {
      rowsByPath.set(p, await getRows(p));
      fetchedAtByPath.set(p, Date.now());
    }));
    saveCache();
    renderTree();
  }

  function activeRefreshPaths() {
    const set = new Set(['.']);
    for (const p of expandedPaths) set.add(p);
    return [...set];
  }

  async function refreshTree() {
    const button = panel.querySelector('.solveit-tree-refresh');
    button.disabled = true; setStatus('Refreshing files…');
    try {
      const keys = activeRefreshPaths();
      const results = await Promise.allSettled(keys.map(async p => [p, await getRows(p)]));
      let changed = false;
      results.forEach((res, i) => {
        if (res.status === 'fulfilled') {
          const [p, rows] = res.value;
          const prev = rowsByPath.get(p);
          if (!prev || JSON.stringify(prev) !== JSON.stringify(rows)) changed = true;
          rowsByPath.set(p, rows); fetchedAtByPath.set(p, Date.now());
        } else if (keys[i] !== '.') {
          rowsByPath.delete(keys[i]); fetchedAtByPath.delete(keys[i]); expandedPaths.delete(keys[i]);
          changed = true;
        }
      });
      saveCache();
      if (changed || !tree) renderTree();
      setStatus('');
    } catch (err) { setStatus(`Could not refresh files: ${err.message || err}`); }
    finally { button.disabled = false; }
  }

  async function ensureCurrentDirLoaded() {
    const {dir} = getCurrentContext();
    const chain = ancestorsOf(dir);
    const missing = [];
    for (const p of chain) {
      if (p !== '.') expandedPaths.add(p);
      if (!rowsByPath.has(p)) missing.push(p);
    }
    if (missing.length) {
      const results = await Promise.all(missing.map(async p => [p, await getRows(p)]));
      const now = Date.now();
      for (const [p, rows] of results) {
        rowsByPath.set(p, rows);
        fetchedAtByPath.set(p, now);
      }
    }
    saveCache();
  }

  function openNode(node) {
    if (!node.href) return;
    // On a phone the drawer covers the content: get it out of the way (and don't restore it after navigation).
    if (window.craftMobile?.active() && panel?.classList.contains('open')) closePanel();
    if (node.kind === 'dialog') { location.href = node.href; return; }
    const url = new URL(node.href, location.href); url.pathname = '/file_editor_';
    let overlay = qs('#solveit-temp-editor');
    if (!overlay) {
      const {root: ov, close, frame} = mountTpl('ft_temp_editor', {refs: {close: 'button', frame: 'iframe'}});
      overlay = ov;
      close.onclick = () => { overlay.classList.remove('open'); frame.src = 'about:blank'; };
      document.body.append(overlay);
      on(frame, 'load', () => {
        on(frame.contentWindow, 'keydown', e => {
          if (window.solveitHotkeys?.dispatch(e)) { e.preventDefault(); e.stopImmediatePropagation(); }
        }, true);
        // Phone: the overlay bar already names the file and closes it, so give Monaco the page's own header room too.
        const fd = frame.contentDocument;
        if (window.craftMobile?.active() && fd?.head && !fd.getElementById('craft-mobile-file-editor')) {
          const st = fd.createElement('style'); st.id = 'craft-mobile-file-editor';
          st.textContent = 'body > div.h-screen > header { display: none !important; } body > div.h-screen { padding: 0 !important; gap: 0 !important; } #file_editor { border: 0 !important; }';
          fd.head.append(st);
          frame.contentWindow.dispatchEvent(new Event('resize'));
        }
      });
    }
    qs('strong', overlay).textContent = url.searchParams.get('fp') || 'File editor';
    overlay.classList.add('open'); qs('iframe', overlay).src = url.href;
  }

  function handleTreeClick(e) {
    if (restoring || e.target.closest('.solveit-tree-create')) return;
    const li = e.target.closest('li[data-node-data]');
    if (!li) return;
    const node = JSON.parse(li.dataset.nodeData), expander = e.target.closest('.treeview-expander');
    const moreBtn = e.target.closest('.solveit-tree-row-more');
    if (moreBtn) {
      e.preventDefault(); e.stopPropagation();
      const content = moreBtn.closest('.treeview-node-content');
      showMenu({target: content, currentTarget: moreBtn, clientX: 0, clientY: 0, preventDefault() {}, stopPropagation() {}});
      return;
    }
    const upBtn = e.target.closest('.solveit-tree-row-upload');
    if (upBtn && node.kind === 'directory') {
      e.preventDefault(); e.stopPropagation();
      openUploadPop(node.path, upBtn);
      return;
    }
    if (node.kind === 'directory' && !expander) {
      e.preventDefault(); e.stopPropagation(); li.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click(); return;
    }
    if (node.kind === 'directory' && expander) queueMicrotask(() => {
      if (!li.isConnected) return;
      const open = li.classList.contains('expanded'), icon = li.querySelector(':scope > .treeview-node-content > img.solveit-tree-icon');
      li.querySelector(':scope > .treeview-node-content')?.setAttribute('aria-expanded', String(open));
      if (open) { expandedPaths.add(node.path); if (icon) icon.src = '/static/.craft-assets/vscode-icons/default_folder_opened.svg'; loadFolder(node.path); }
      else { expandedPaths.delete(node.path); if (icon) icon.src = '/static/.craft-assets/vscode-icons/default_folder.svg'; }
      saveCache();
    });
    else if (node.href) openNode(node);
  }

  /* ── File operations: Create, Delete, Copy/Paste, Drag-Move ────────── */
  function validateName(raw, kind) {
    const trimmed = raw.trim();
    if (!trimmed) throw new Error('Enter a name.');
    if (trimmed === '.' || trimmed === '..' || /[\\/]/.test(trimmed)) throw new Error('Name cannot be "." or ".." or contain slashes.');
    if (kind === 'dialog') return trimmed.endsWith('.ipynb') ? trimmed : `${trimmed}.ipynb`;
    return trimmed;
  }

  function uniqueName(existingSet, filename, kind) {
    if (!existingSet.has(filename)) return filename;
    let stem = filename, ext = '';
    if (kind !== 'directory') {
      const dot = filename.lastIndexOf('.');
      if (dot > 0) { stem = filename.slice(0, dot); ext = filename.slice(dot); }
    }
    let candidate = `${stem} copy${ext}`, n = 2;
    while (existingSet.has(candidate)) candidate = `${stem} copy ${n++}${ext}`;
    return candidate;
  }

  async function withBusy(label, fn) {
    if (busy) { setStatus('Wait for the current file action to finish…'); return; }
    busy = true; setStatus(label);
    try { return await fn(); }
    catch (err) { setStatus(err.message || String(err)); }
    finally { busy = false; }
  }

  async function createItem(targetDir, kind, filename) {
    const existing = await getRows(targetDir);
    if (existing.some(r => r.filename === filename)) throw new Error(`'${filename}' already exists in this folder.`);
  if (kind === 'directory') {
    await uploadBlob(targetDir, '__mkdir__.zip', makeEmptyDirZip(filename), true);
  } else if (kind === 'dialog') {
    const nb = JSON.stringify({cells: [], metadata: {solveit: {mode: 'learning'}}, nbformat: 4, nbformat_minor: 5}, null, 1);
    await uploadBlob(targetDir, filename, new Blob([nb], {type: 'application/json'}), false);
  } else {
    await uploadBlob(targetDir, filename, new Blob([''], {type: 'text/plain'}), false);
  }
  }

  let createDraft;
  function dismissCreate(restoreFocus = false) {
    if (!createDraft) return;
    const {form, dir} = createDraft;
    createDraft = null;
    form.remove();
    if (restoreFocus && !focusPath(dir)) panel?.querySelector('.solveit-tree-filter')?.focus();
  }

  function mountCreate() {
    if (!createDraft) return;
    const {dir, form} = createDraft;
    if (dir === '.') panel.querySelector('#solveit-file-tree').prepend(form);
    else findNode(dir)?.querySelector(':scope > .treeview-node-content')?.after(form);
  }

  function beginCreate(dir, kind) {
    if (busy) { setStatus('Wait for the current file action to finish…'); return; }
    dismissCreate();
    const label = {dialog: 'New dialog name', file: 'New file name', directory: 'New folder name', auto: 'New name (.ipynb by default)'}[kind];
    const {root: form, input, error} = mountTpl('ft_create_form', {refs: {input: 'input', error: '[role="alert"]'}});
    input.placeholder = label;
    input.setAttribute('aria-label', label);
    const setError = msg => { error.textContent = msg; input.setAttribute('aria-invalid', String(!!msg)); };
    const draft = createDraft = {dir, form, input};
    on(form, 'keydown', e => {
      e.stopPropagation();
      if (e.key === 'Escape' && !e.isComposing) { e.preventDefault(); if (!input.readOnly) dismissCreate(true); }
    });
    on(input, 'input', () => setError(''));
    on(form, 'submit', async e => {
      e.preventDefault();
      if (input.readOnly) return;
      if (busy) { setError('Wait for the current file action to finish.'); return; }
      const raw = input.value.trim();
      const resolvedKind = kind === 'auto' ? (/^.+\.[^.]+$/.test(raw) && !raw.endsWith('.ipynb') ? 'file' : 'dialog') : kind;
      let filename;
      try { filename = validateName(raw, resolvedKind); }
      catch (err) { setError(err.message); return; }
      setError('');
      input.readOnly = busy = true;
      setStatus(`Creating '${filename}'…`);
      try {
        await createItem(dir, resolvedKind, filename);
      } catch (err) {
        setStatus('');
        setError(err.message || String(err));
        if (createDraft === draft) input.focus();
        return;
      } finally { input.readOnly = false; busy = false; }
      if (createDraft === draft) dismissCreate();
      if (dir !== '.') expandedPaths.add(dir);
      try { await refreshDirs(dir); }
      catch (err) { setStatus(`Created '${filename}', but could not refresh: ${err.message || err}`); return; }
      if (!createDraft && panel?.classList.contains('open')) focusPath(joinPath(dir, filename));
      setStatus(hideDots && filename.startsWith('.') ? `Created '${filename}' (hidden by .* filter)` : `Created '${filename}'`);
    });
    const folder = dir === '.' ? null : findNode(dir);
    if (folder && !folder.classList.contains('expanded')) folder.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click();
    mountCreate();
    input.focus();
  }

  async function deleteNode(node) {
    const {dlgPath} = getCurrentContext();
    const affectsActive = dlgPath && (dlgPath === node.path || (node.kind === 'directory' && dlgPath.startsWith(node.path + '/')));
    const msg = node.kind === 'directory'
      ? `Delete folder '${node.name}' and all its contents? This cannot be undone.`
      : `Delete '${node.filename || node.name}'? This cannot be undone.`;
    if (!confirm(affectsActive ? `${msg}\n\nNote: This includes the currently open dialog.` : msg)) return;
    await withBusy(`Deleting '${node.filename || node.name}'…`, async () => {
      await deleteRecursive(node);
      if (node.kind === 'directory') purgeDirCache(node.path);
      const clip = getClipboard();
      if (clip && (clip.path === node.path || (node.kind === 'directory' && clip.path.startsWith(node.path + '/')))) {
        setClipboard(null);
      }
      await refreshDirs(node.parentPath || '.');
      setStatus(`Deleted '${node.filename || node.name}'`);
      if (affectsActive) location.href = '/?at=.';
    });
  }

  function copyNode(node) {
    setClipboard({
      path: node.path,
      filename: node.filename || baseOf(node.path),
      name: node.name,
      kind: node.kind,
      dlUrl: node.dlUrl || staticUrl(node.path),
      parentPath: node.parentPath || parentOf(node.path),
    });
    setStatus(`Copied '${node.filename || node.name}'`);
  }

  function canPaste(src, destDir) {
    if (!src) return false;
    const d = normPath(destDir);
    if (src.kind === 'directory' && (d === src.path || d.startsWith(src.path + '/'))) return false;
    return true;
  }

  async function pasteNodeInto(src, destDir) {
    const targetDir = normPath(destDir);
    if (!canPaste(src, targetDir)) { setStatus('Cannot paste a folder into itself.'); return; }
    await withBusy(`Pasting '${src.filename}'…`, async () => {
      const existing = await getRows(targetDir);
      const existingNames = new Set(existing.map(r => r.filename));
      const targetName = uniqueName(existingNames, src.filename, src.kind);
      const bytes = await downloadBytes(src);
      if (src.kind === 'directory') {
        const zipBlob = targetName === src.filename
          ? new Blob([bytes], {type: 'application/zip'})
          : renameZipTopDir(bytes, src.filename, targetName);
        await uploadBlob(targetDir, `${targetName}.zip`, zipBlob, true);
      } else {
        await uploadBlob(targetDir, targetName, new Blob([bytes]), false);
      }
      if (targetDir !== '.') expandedPaths.add(targetDir);
      await refreshDirs(targetDir);
      const newPath = joinPath(targetDir, targetName);
      focusPath(newPath);
      setStatus(`Pasted '${targetName}'`);
    });
  }

  function canDrop(src, destDir) {
    if (!src || src.kind === 'loading') return false;
    const d = normPath(destDir);
    if ((src.parentPath || parentOf(src.path)) === d) return false;
    if (src.kind === 'directory' && (d === src.path || d.startsWith(src.path + '/'))) return false;
    return true;
  }

  async function moveNode(src, destDir) {
    const targetDir = normPath(destDir);
    if (!canDrop(src, targetDir)) return;
    const {dlgPath} = getCurrentContext();
    const affectsActive = dlgPath && (dlgPath === src.path || (src.kind === 'directory' && dlgPath.startsWith(src.path + '/')));
    if (affectsActive && !confirm(`'${src.filename}' includes the currently open dialog. Moving it will reload at the new location. Continue?`)) return;

    await withBusy(`Moving '${src.filename}' to ${targetDir === '.' ? '~/' : targetDir + '/'}…`, async () => {
      const destBefore = await getRows(targetDir);
      if (destBefore.some(r => r.filename === src.filename)) {
        throw new Error(`Cannot move: '${src.filename}' already exists in ${targetDir === '.' ? '~/' : targetDir}`);
      }
      const bytes = await downloadBytes(src);
      if (src.kind === 'directory') {
        await uploadBlob(targetDir, `${src.filename}.zip`, new Blob([bytes], {type: 'application/zip'}), true);
      } else {
        await uploadBlob(targetDir, src.filename, new Blob([bytes]), false);
      }
      // Verify destination before deleting source
      const destAfter = await getRows(targetDir);
      const created = destAfter.find(r => r.filename === src.filename && r.kind === src.kind);
      if (!created) throw new Error(`Move aborted: '${src.filename}' did not appear at destination`);

      await deleteRecursive(src);
      const wasExpanded = src.kind === 'directory' && expandedPaths.has(src.path);
      if (src.kind === 'directory') purgeDirCache(src.path);
      const newPath = joinPath(targetDir, src.filename);
      if (wasExpanded) expandedPaths.add(newPath);
      if (targetDir !== '.') expandedPaths.add(targetDir);

      const clip = getClipboard();
      if (clip && (clip.path === src.path || (src.kind === 'directory' && clip.path.startsWith(src.path + '/')))) {
        setClipboard(null);
      }
      rowsByPath.set(targetDir, destAfter); fetchedAtByPath.set(targetDir, Date.now());
      await refreshDirs(src.parentPath || '.', ...(wasExpanded ? [newPath] : []));
      focusPath(newPath);
      setStatus(`Moved '${src.filename}' to ${targetDir === '.' ? '~/' : targetDir + '/'}`);

      if (affectsActive) {
        const nextDlg = dlgPath === src.path ? newPath : newPath + dlgPath.slice(src.path.length);
        location.href = `/dialog_?name=${encodeURIComponent(nextDlg)}`;
      }
    });
  }

  async function renameNode(node) {
    if (!node || node.kind === 'loading') return;
    const oldName = node.filename || baseOf(node.path);
    const dir = node.parentPath || parentOf(node.path);
    const raw = prompt(`Rename '${oldName}' to:`, oldName);
    if (raw === null) return;
    const newName = raw.trim();
    if (!newName || newName === oldName) return;
    if (newName.includes('/') || newName.includes('\\') || newName === '.' || newName === '..') {
      setStatus('Invalid name: cannot contain slashes or be "." / "..".');
      return;
    }
    const {dlgPath} = getCurrentContext();
    const affectsActive = dlgPath && (dlgPath === node.path || (node.kind === 'directory' && dlgPath.startsWith(node.path + '/')));
    if (affectsActive && !confirm(`'${oldName}' includes the currently open dialog. Renaming it will reload at the new path. Continue?`)) return;

    await withBusy(`Renaming '${oldName}' to '${newName}'…`, async () => {
      const siblings = await getRows(dir);
      if (siblings.some(r => r.filename === newName)) {
        throw new Error(`'${newName}' already exists in ${dir === '.' ? '~/' : dir}`);
      }
      const bytes = await downloadBytes(node);
      if (node.kind === 'directory') {
        const zipBlob = renameZipTopDir(bytes, oldName, newName);
        await uploadBlob(dir, `${newName}.zip`, zipBlob, true);
      } else {
        await uploadBlob(dir, newName, new Blob([bytes]), false);
      }
      const after = await getRows(dir);
      const created = after.find(r => r.filename === newName);
      if (!created) throw new Error(`Rename aborted: '${newName}' did not appear`);

      await deleteRecursive(node);
      const wasExpanded = node.kind === 'directory' && expandedPaths.has(node.path);
      if (node.kind === 'directory') purgeDirCache(node.path);
      const newPath = joinPath(dir, newName);
      if (wasExpanded) expandedPaths.add(newPath);

      const clip = getClipboard();
      if (clip && (clip.path === node.path || (node.kind === 'directory' && clip.path.startsWith(node.path + '/')))) {
        setClipboard(null);
      }
      rowsByPath.set(dir, after); fetchedAtByPath.set(dir, Date.now());
      await refreshDirs(dir, ...(wasExpanded ? [newPath] : []));
      focusPath(newPath);
      setStatus(`Renamed '${oldName}' to '${newName}'`);

      if (affectsActive) {
        const nextDlg = dlgPath === node.path ? newPath : newPath + dlgPath.slice(node.path.length);
        location.href = `/dialog_?name=${encodeURIComponent(nextDlg)}`;
      }
    });
  }

  /* ── Upload (files & folders, via drag-and-drop or Upload popover) ─── */
  function snapshotDataTransfer(dt) {
    const entries = dt?.items
      ? [...dt.items].filter(it => it.kind === 'file').map(it => it.webkitGetAsEntry?.()).filter(Boolean)
      : [];
    const files = dt?.files ? [...dt.files] : [];
    return {entries, files};
  }

  function readAllDirEntries(dirEntry) {
    return new Promise((resolve, reject) => {
      const reader = dirEntry.createReader(), all = [];
      const next = () => reader.readEntries(batch => {
        if (!batch.length) resolve(all);
        else { all.push(...batch); next(); }
      }, reject);
      next();
    });
  }

  async function collectFromEntries(entries) {
    const items = [];
    for (const entry of entries) {
      if (!entry) continue;
      if (entry.isFile) {
        const file = await new Promise((res, rej) => entry.file(res, rej));
        items.push({kind: 'file', name: entry.name || file.name, file});
      } else if (entry.isDirectory) {
        const dirs = [], files = [];
        const walk = async (dirEntry, relDir) => {
          const children = await readAllDirEntries(dirEntry);
          for (const child of children) {
            if (child.isDirectory) {
              const nextRel = relDir ? `${relDir}/${child.name}` : child.name;
              dirs.push(nextRel);
              await walk(child, nextRel);
            } else if (child.isFile) {
              const file = await new Promise((res, rej) => child.file(res, rej));
              files.push({relDir, name: child.name || file.name, file});
            }
          }
        };
        await walk(entry, '');
        items.push({kind: 'directory', name: entry.name, dirs, files});
      }
    }
    return items;
  }

  function collectFromFileList(fileList) {
    const files = [...(fileList || [])];
    const dirsByTop = new Map(), items = [];
    for (const file of files) {
      const rel = (file.webkitRelativePath || '').replace(/^\/+/, '');
      if (rel && rel.includes('/')) {
        const parts = rel.split('/'), top = parts[0];
        if (!dirsByTop.has(top)) {
          const rec = {kind: 'directory', name: top, dirs: new Set(), files: []};
          dirsByTop.set(top, rec);
          items.push(rec);
        }
        const rec = dirsByTop.get(top);
        let acc = '';
        for (const seg of parts.slice(1, -1)) { acc = acc ? `${acc}/${seg}` : seg; rec.dirs.add(acc); }
        rec.files.push({relDir: acc, name: parts[parts.length - 1], file});
      } else {
        items.push({kind: 'file', name: file.name, file});
      }
    }
    return items.map(it => it.kind === 'directory' ? {...it, dirs: [...it.dirs]} : it);
  }

  async function handleDroppedTransfer(destDir, snap) {
    try {
      const items = snap.entries.length ? await collectFromEntries(snap.entries) : collectFromFileList(snap.files);
      await uploadItemsTo(destDir, items);
    } catch (err) {
      setStatus('Upload failed: ' + (err.message || err));
    }
  }

  async function uploadItemsTo(targetDir, items) {
    const dest = normPath(targetDir);
    if (!items.length) { setStatus('No files found to upload.'); return; }
    await withBusy(`Uploading to ${dest === '.' ? '~/' : dest + '/'}…`, async () => {
      const existing = await getRows(dest);
      const existingNames = new Set(existing.map(r => r.filename));
      const createdPaths = [], dirsToRefresh = new Set([dest]);
      let uploadedFiles = 0, createdDirs = 0;

      for (const item of items) {
        if (item.kind === 'file') {
          const targetName = uniqueName(existingNames, item.name, 'file');
          existingNames.add(targetName);
          setStatus(`Uploading '${targetName}'…`);
          await uploadBlob(dest, targetName, item.file, false);
          uploadedFiles++;
          createdPaths.push(joinPath(dest, targetName));
        } else {
          const targetDirName = uniqueName(existingNames, item.name, 'directory');
          existingNames.add(targetDirName);
          const rootCreated = joinPath(dest, targetDirName);
          setStatus(`Creating folder '${targetDirName}'…`);
          await uploadBlob(dest, '__mkdir__.zip', makeEmptyDirZip(targetDirName), true);
          createdDirs++;
          createdPaths.push(rootCreated);

          const sortedDirs = [...new Set(item.dirs)].sort((a, b) => a.split('/').length - b.split('/').length);
          for (const relSub of sortedDirs) {
            const parentSub = parentOf(relSub);
            const parentFull = parentSub === '.' ? rootCreated : joinPath(rootCreated, parentSub);
            await uploadBlob(parentFull, '__mkdir__.zip', makeEmptyDirZip(baseOf(relSub)), true);
            createdDirs++;
          }
          for (const f of item.files) {
            const fileParent = f.relDir ? joinPath(rootCreated, f.relDir) : rootCreated;
            setStatus(`Uploading '${f.name}'…`);
            await uploadBlob(fileParent, f.name, f.file, false);
            uploadedFiles++;
          }
          if (rowsByPath.has(rootCreated)) dirsToRefresh.add(rootCreated);
        }
      }

      if (dest !== '.') expandedPaths.add(dest);
      await refreshDirs(...dirsToRefresh);
      if (createdPaths.length) focusPath(createdPaths[createdPaths.length - 1]);
      const parts = [];
      if (uploadedFiles) parts.push(`${uploadedFiles} file${uploadedFiles === 1 ? '' : 's'}`);
      if (createdDirs) parts.push(`${createdDirs} folder${createdDirs === 1 ? '' : 's'}`);
      const hiddenCount = hideDots ? createdPaths.filter(p => baseOf(p).startsWith('.')).length : 0;
      const suffix = hiddenCount ? ' (hidden by .* filter)' : '';
      setStatus(`Uploaded ${parts.join(', ')} to ${dest === '.' ? '~/' : dest + '/'}${suffix}`);
    });
  }

  function closeUploadPop() {
    if (!uploadPop) return false;
    uploadPop.remove(); uploadPop = null;
    return true;
  }

  function openUploadPop(targetDir, anchorEl = null) {
    const dest = normPath(targetDir);
    closeMenu(); closeUploadPop();
    const {root: pop, dest: destEl, files: filesInput, dir: dirInput, drop: dropZone, close: closeBtn, pickFiles, pickDir} = mountTpl('ft_upload_pop', {refs: {
      dest: '.solveit-tree-upload-dest', files: '.solveit-tree-upload-input-files',
      dir: '.solveit-tree-upload-input-dir', drop: '.solveit-tree-upload-drop',
      close: '.solveit-tree-upload-x', pickFiles: '.solveit-tree-upload-pick-files',
      pickDir: '.solveit-tree-upload-pick-dir',
    }});
    uploadPop = pop;
    pop.setAttribute('aria-label', `Upload to ${dest === '.' ? '~/' : dest}`);
    const labelPath = dest === '.' ? '~/' : `~/${dest}/`;
    destEl.textContent = `Upload to ${labelPath}`; destEl.title = labelPath;

    closeBtn.onclick = () => closeUploadPop();
    pickFiles.onclick = () => { filesInput.value = ''; filesInput.click(); };
    pickDir.onclick = () => { dirInput.value = ''; dirInput.click(); };

    const onPicked = input => () => {
      const items = collectFromFileList(input.files);
      closeUploadPop();
      if (items.length) uploadItemsTo(dest, items);
    };
    on(filesInput, 'change', onPicked(filesInput));
    on(dirInput, 'change', onPicked(dirInput));

    on(dropZone, {
      dragover: e => {
        e.preventDefault(); e.stopPropagation();
        e.dataTransfer.dropEffect = 'copy';
        dropZone.classList.add('dragover');
      },
      dragleave: e => {
        if (!dropZone.contains(e.relatedTarget)) dropZone.classList.remove('dragover');
      },
      drop: e => {
        e.preventDefault(); e.stopPropagation();
        dropZone.classList.remove('dragover');
        const snap = snapshotDataTransfer(e.dataTransfer);
        closeUploadPop();
        handleDroppedTransfer(dest, snap);
      },
    });

    panel.append(pop);
    const b = anchorEl?.getBoundingClientRect?.() || panel.getBoundingClientRect();
    const r = pop.getBoundingClientRect();
    pop.style.left = Math.max(4, Math.min(b.left, innerWidth - r.width - 4)) + 'px';
    pop.style.top = Math.max(4, Math.min(b.bottom + 4, innerHeight - r.height - 4)) + 'px';
    pop.querySelector('.solveit-tree-upload-pick-files')?.focus();
  }

  function collapseAllFolders() {
    closeMenu(); closeUploadPop(); dismissCreate();
    expandedPaths.clear();
    saveCache();
    if (tree) renderTree();
    setStatus('Collapsed all folders');
  }

  /* ── Context menu ──────────────────────────────────────────────────── */
  function closeMenu() { if (!menu) return false; menu.remove(); menu = null; return true; }

  async function copyNodePath(node) {
    const path = `/app/data/${node.path}`;
    try {
      await navigator.clipboard.writeText(path);
      setStatus(`Copied path: ${path}`);
    } catch (err) { setStatus(`Could not copy path: ${err.message}`); }
  }

  function showMenu(e, forcedDir = null) {
    if (e.target.closest('.solveit-tree-create')) return;
    const li = forcedDir ? null : e.target.closest('li[data-node-data]');
    const node = li ? JSON.parse(li.dataset.nodeData) : null;
    if (node?.kind === 'loading') return;
    e.preventDefault(); e.stopPropagation(); closeMenu();

    const clip = getClipboard();
    menu = window.__craftCMCore.mountTpl('ft_menu');
    const item = (label, fn, {disabled = false, title = '', danger = false} = {}) => {
      menu.append(window.__craftCMCore.el('button', {
        type: 'button', role: 'menuitem', disabled, title: title || null, cls: danger ? 'danger' : null,
        onClick: ev => { ev.stopPropagation(); closeMenu(); fn(); },
      }, label));
    };
    const sep = () => { menu.append(window.__craftCMCore.el('hr', {cls: 'solveit-tree-menu-sep'})); };

    if (!node || node.kind === 'directory') {
      const targetDir = forcedDir ? normPath(forcedDir) : (node ? node.path : '.');
      item('New Dialog…', () => beginCreate(targetDir, 'dialog'));
      item('New File…', () => beginCreate(targetDir, 'file'));
      item('New Folder…', () => beginCreate(targetDir, 'directory'));
      item('Upload…', () => openUploadPop(targetDir, li || e.currentTarget));
      sep();
      if (node) {
        item('Rename…', () => renameNode(node));
        item('Copy', () => copyNode(node));
        item('Copy Path', () => copyNodePath(node));
      }
      item(clip ? `Paste '${clip.filename}'` : 'Paste', () => pasteNodeInto(clip, targetDir), {
        disabled: !clip || !canPaste(clip, targetDir),
        title: !clip ? 'Copy a file, dialog, or folder first' : (!canPaste(clip, targetDir) ? 'Cannot paste a folder into itself' : ''),
      });
      if (node) {
        sep();
        item('Delete…', () => deleteNode(node), {danger: true});
      }
    } else {
      const targetDir = node.parentPath || '.';
      item('Open', () => openNode(node), {disabled: !node.href});
      item('Upload…', () => openUploadPop(targetDir, li || e.currentTarget));
      sep();
      item('Rename…', () => renameNode(node));
      item('Copy', () => copyNode(node));
      item('Copy Path', () => copyNodePath(node));
      item(clip ? `Paste '${clip.filename}'` : 'Paste', () => pasteNodeInto(clip, targetDir), {
        disabled: !clip || !canPaste(clip, targetDir),
        title: !clip ? 'Copy a file, dialog, or folder first' : (!canPaste(clip, targetDir) ? 'Cannot paste a folder into itself' : ''),
      });
      item('Duplicate', () => pasteNodeInto(node, targetDir));
      sep();
      item('Delete…', () => deleteNode(node), {danger: true});
    }

    menu.addEventListener('keydown', ev => {
      if (ev.key !== 'ArrowDown' && ev.key !== 'ArrowUp') return;
      ev.preventDefault(); ev.stopPropagation();
      const btns = [...menu.querySelectorAll('button:not(:disabled)')];
      if (!btns.length) return;
      const i = btns.indexOf(document.activeElement);
      btns[(i + (ev.key === 'ArrowDown' ? 1 : -1) + btns.length) % btns.length].focus();
    });

    panel.append(menu);
    const anchor = li || e.currentTarget;
    const at = (e.clientX || e.clientY) ? {x: e.clientX, y: e.clientY} : (b => ({x: b.left + 24, y: b.bottom}))(anchor.getBoundingClientRect());
    const r = menu.getBoundingClientRect();
    menu.style.left = Math.max(4, Math.min(at.x, innerWidth - r.width - 4)) + 'px';
    menu.style.top = Math.max(4, Math.min(at.y, innerHeight - r.height - 4)) + 'px';
    menu.querySelector('button:not(:disabled)')?.focus();
  }

  /* ── Drag and drop ─────────────────────────────────────────────────── */
  function clearDropState() {
    if (dropHighlightEl) { dropHighlightEl.classList.remove('solveit-tree-drop-target'); dropHighlightEl = null; }
    clearTimeout(hoverExpandTimer); hoverExpandTimer = null; hoverExpandPath = null;
  }

  function resolveDropTarget(targetEl) {
    const rootbar = panel?.querySelector('.solveit-tree-rootbar');
    if (targetEl.closest?.('.solveit-tree-rootbar')) return {destDir: '.', highlightEl: rootbar, dirLi: null};
    const li = targetEl.closest?.('li[data-node-data]');
    if (!li) return {destDir: '.', highlightEl: rootbar, dirLi: null};
    const node = JSON.parse(li.dataset.nodeData);
    if (node.kind === 'loading') return null;
    if (node.kind === 'directory') {
      return {destDir: node.path, highlightEl: li.querySelector(':scope > .treeview-node-content'), dirLi: li};
    }
    const destDir = node.parentPath || '.';
    const parentLi = destDir === '.' ? null : findNode(destDir);
    return {
      destDir,
      highlightEl: parentLi ? parentLi.querySelector(':scope > .treeview-node-content') : rootbar,
      dirLi: parentLi,
    };
  }

  function wireDragAndDrop(treeEl, rootbar) {
    treeEl.addEventListener('dragstart', e => {
      const li = e.target.closest('li[data-node-data]');
      if (!li) return;
      const node = JSON.parse(li.dataset.nodeData);
      if (node.kind === 'loading') { e.preventDefault(); return; }
      closeMenu();
      draggedNode = node;
      li.classList.add('solveit-tree-dragging');
      e.dataTransfer.effectAllowed = 'move';
      try {
        e.dataTransfer.setData('application/x-solveit-node', JSON.stringify(node));
        e.dataTransfer.setData('text/plain', node.path);
      } catch {}
    });

    const isExternalFileDrag = e => !draggedNode && [...(e.dataTransfer?.types || [])].includes('Files');

    const onDragOver = e => {
      const external = isExternalFileDrag(e);
      if (!draggedNode && !external) return;
      if (e.target.closest?.('.solveit-tree-upload-pop')) return;
      const target = resolveDropTarget(e.target);
      if (!target || (!external && !canDrop(draggedNode, target.destDir))) {
        clearDropState();
        e.dataTransfer.dropEffect = 'none';
        return;
      }
      e.preventDefault();
      e.dataTransfer.dropEffect = external ? 'copy' : 'move';
      if (dropHighlightEl !== target.highlightEl) {
        if (dropHighlightEl) dropHighlightEl.classList.remove('solveit-tree-drop-target');
        dropHighlightEl = target.highlightEl;
        dropHighlightEl?.classList.add('solveit-tree-drop-target');
      }
      if (target.dirLi && !target.dirLi.classList.contains('expanded') && target.destDir !== '.') {
        if (hoverExpandPath !== target.destDir) {
          clearTimeout(hoverExpandTimer);
          hoverExpandPath = target.destDir;
          const dirLi = target.dirLi;
          hoverExpandTimer = setTimeout(() => {
            if (dirLi.isConnected && !dirLi.classList.contains('expanded')) {
              dirLi.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click();
            }
          }, 600);
        }
      } else {
        clearTimeout(hoverExpandTimer); hoverExpandTimer = null; hoverExpandPath = null;
      }
    };

    const onDrop = e => {
      const external = isExternalFileDrag(e);
      if (!draggedNode && !external) return;
      if (e.target.closest?.('.solveit-tree-upload-pop')) return;
      e.preventDefault(); e.stopPropagation();
      const target = resolveDropTarget(e.target);
      clearDropState();
      if (external) {
        const snap = snapshotDataTransfer(e.dataTransfer);
        if (target) handleDroppedTransfer(target.destDir, snap);
        return;
      }
      const src = draggedNode;
      panel?.querySelectorAll('li.solveit-tree-dragging').forEach(el => el.classList.remove('solveit-tree-dragging'));
      draggedNode = null;
      if (target && canDrop(src, target.destDir)) moveNode(src, target.destDir);
    };

    const onDragEnd = () => {
      clearDropState();
      panel?.querySelectorAll('li.solveit-tree-dragging').forEach(el => el.classList.remove('solveit-tree-dragging'));
      draggedNode = null;
    };

    const onDragLeave = e => {
      if (!panel.contains(e.relatedTarget)) clearDropState();
    };

    on(panel, {dragover: onDragOver, drop: onDrop, dragleave: onDragLeave});
    on(treeEl, 'dragend', onDragEnd);
  }

  /* ── Panel construction & lifecycle ────────────────────────────────── */
  function updateDotsBtn() {
    const dots = qs('.solveit-tree-dots', panel);
    if (!dots) return;
    const lbl = hideDots ? 'Hidden files are hidden (click to show)' : 'Showing all files (click to hide dot files)';
    patch(dots, {'aria-pressed': String(hideDots), title: lbl, 'aria-label': lbl});
  }

  function makePanel() {
    if (panel) return panel;
    const r = mountTpl('ft_panel', {refs: {
      brand: '.solveit-tree-brand', btnNew: '.solveit-tree-new', btnUpload: '.solveit-tree-upload',
      btnCollapse: '.solveit-tree-collapse', btnRefresh: '.solveit-tree-refresh', btnDots: '.solveit-tree-dots',
      btnHome: '.solveit-tree-home', filter: '.solveit-tree-filter', treeEl: '#solveit-file-tree',
      rootbar: '.solveit-tree-rootbar', resizer: '.solveit-tree-resizer',
    }});
    panel = r.root;
    const {treeEl, rootbar, resizer} = r;

    dlgNameNode = qs('nav.monster-navbar h2#dlg-name');
    if (dlgNameNode) {
      dlgNameOrigParent = dlgNameNode.parentNode;
      dlgNameOrigNext = dlgNameNode.nextSibling;
      r.brand.append(dlgNameNode);
    }
    on(panel, 'click', e => e.stopPropagation());
    r.btnNew.onclick = () => beginCreate(lastTreeDir || getCurrentContext().dir, 'auto');
    r.btnUpload.onclick = e => openUploadPop(lastTreeDir || getCurrentContext().dir, e.currentTarget);
    r.btnCollapse.onclick = collapseAllFolders;
    r.btnRefresh.onclick = refreshTree;
    updateDotsBtn();
    r.btnDots.onclick = () => {
      hideDots = !hideDots;
      updateDotsBtn();
      store.set(DOTS_KEY, hideDots ? '1' : '0');
      if (tree) renderTree();
    };
    r.btnHome.onclick = () => { location.href = '/?at=.'; };
    window.solveitHotkeys?.refreshChips();
    let filterTimer = null;
    on(r.filter, 'input', e => {
      filterText = e.target.value;
      clearTimeout(filterTimer);
      if (!filterText) { tree?.search(''); return; }
      filterTimer = setTimeout(() => { tree?.search(filterText); }, 80);
    });
    rootbar.querySelector('.solveit-tree-row-upload').onclick = e => {
      e.stopPropagation();
      openUploadPop('.', e.currentTarget);
    };
    rootbar.querySelector('.solveit-tree-row-more').onclick = e => {
      e.stopPropagation();
      const b = e.currentTarget;
      showMenu({target: b, currentTarget: b, clientX: 0, clientY: 0, preventDefault() {}, stopPropagation() {}}, '.');
    };
    treeEl.addEventListener('focusin', e => {
      const li = e.target.closest?.('li[data-node-data]');
      if (!li) return;
      const node = JSON.parse(li.dataset.nodeData);
      if (node.kind === 'loading') return;
      lastTreeDir = node.kind === 'directory' ? node.path : (node.parentPath || '.');
    });
    treeEl.addEventListener('contextmenu', e => showMenu(e));
    rootbar.addEventListener('contextmenu', e => showMenu(e, '.'));
    panel.addEventListener('contextmenu', e => {
      if (e.target === panel || e.target.matches('.solveit-tree-status')) showMenu(e, '.');
    });
    /* iOS Safari never fires contextmenu for a touch hold, so hold-to-open is done by hand.
       Android fires both: whichever comes first wins and the other is swallowed. */
    let lpTimer = null, lpStart = null, lpFiredAt = 0;
    const lpCancel = () => { clearTimeout(lpTimer); lpTimer = null; };
    panel.addEventListener('touchstart', e => {
      lpCancel();
      if (e.touches.length !== 1) return;
      const t = e.target, onRoot = rootbar.contains(t);
      if (t.closest('button, input, .solveit-tree-create') || !(onRoot || t.closest('li[data-node-data]'))) return;
      const {clientX, clientY} = e.touches[0];
      lpStart = {x: clientX, y: clientY};
      lpTimer = setTimeout(() => {
        lpTimer = null; lpFiredAt = Date.now();
        navigator.vibrate?.(10);
        showMenu({target: t, currentTarget: t, clientX, clientY, preventDefault() {}, stopPropagation() {}}, onRoot ? '.' : null);
      }, 500);
    }, {passive: true});
    panel.addEventListener('touchmove', e => {
      const p = e.touches[0];
      if (lpTimer && Math.hypot(p.clientX - lpStart.x, p.clientY - lpStart.y) > 10) lpCancel();
    }, {passive: true});
    panel.addEventListener('touchend', e => { lpCancel(); if (Date.now() - lpFiredAt < 800) e.preventDefault(); });  // the hold must not also open the row
    panel.addEventListener('touchcancel', lpCancel, {passive: true});
    panel.addEventListener('contextmenu', e => {
      lpCancel();
      if (Date.now() - lpFiredAt < 1000) { e.preventDefault(); e.stopImmediatePropagation(); }
    }, true);
    panel.addEventListener('dblclick', e => {
      if (e.target.closest('.solveit-tree-row-upload, .solveit-tree-upload-pop')) return;
      const atRoot = rootbar.contains(e.target);
      const onBlank = e.target === panel || e.target.matches('.solveit-tree-status') || (treeEl.contains(e.target) && !e.target.closest('li[data-node-data], .solveit-tree-create'));
      if (!atRoot && !onBlank) return;
      e.preventDefault(); e.stopPropagation();
      closeMenu();
      beginCreate(atRoot ? '.' : getCurrentContext().dir, 'auto');
    });
    treeEl.addEventListener('click', handleTreeClick, true);
    wireDragAndDrop(treeEl, rootbar);
    const WIDTH_KEY = 'solveit-file-tree:width';
    const applyWidth = px => {
      if (!px) { panel.style.removeProperty('--solveit-tree-w'); return; }
      const clamped = Math.max(200, Math.min(Math.round(px), Math.round(innerWidth * 0.6)));
      panel.style.setProperty('--solveit-tree-w', clamped + 'px');
    };
    const savedW = Number(store.get(WIDTH_KEY)); if (savedW >= 200) applyWidth(savedW);
    on(resizer, {
      pointerdown: e => {
        if (e.button !== 0 || window.craftMobile?.active?.()) return;
        e.preventDefault(); e.stopPropagation();
        resizer.setPointerCapture?.(e.pointerId);
        panel.classList.add('resizing');
        const startX = e.clientX, startW = panel.getBoundingClientRect().width;
        const onMove = ev => applyWidth(startW + (ev.clientX - startX));
        let offDrag;
        const onUp = () => {
          panel.classList.remove('resizing');
          offDrag?.();
          store.set(WIDTH_KEY, Math.round(panel.getBoundingClientRect().width));
        };
        offDrag = on(resizer, {pointermove: onMove, pointerup: onUp, pointercancel: onUp});
      },
      dblclick: e => {
        e.preventDefault(); e.stopPropagation();
        applyWidth(0);
        store.del(WIDTH_KEY);
      },
    });

    panel.inert = true;
    const scroller = qs('.uk-container > .overflow-y-auto');
    if (scroller) {
      workspaceRow = mountTpl('ft_workspace');
      scroller.before(workspaceRow);
      workspaceRow.append(panel, scroller);
    } else document.body.append(panel);
    return panel;
  }

  async function toggle(focus = true) {
    const p = makePanel(), opening = !p.classList.contains('open');
    if (!opening) { closePanel(); return; }
    previousFocus = document.activeElement; p.inert = false; p.classList.add('open');
    saveOpen(true);
    if (!refreshTimer) refreshTimer = setInterval(() => {
      if (panel?.classList.contains('open') && activeRefreshPaths().some(path => Date.now() - (fetchedAtByPath.get(path) || 0) > CACHE_MAX_AGE)) refreshTree();
    }, CACHE_MAX_AGE);
    try {
      setStatus('Loading files…');
      await loadAssets();
      await ensureCurrentDirLoaded();
      renderTree(); setStatus('');
      if (focus && p.classList.contains('open')) {
        const {dir, dlgPath} = getCurrentContext();
        if (!(dlgPath && focusPath(dlgPath)) && !(dir !== '.' && focusPath(dir))) p.querySelector('#solveit-file-tree .treeview-node-content')?.focus();
      }
      if (activeRefreshPaths().some(path => Date.now() - (fetchedAtByPath.get(path) || 0) > CACHE_MAX_AGE)) refreshTree();
    } catch (err) { setStatus('Unable to load files: ' + (err.message || err)); }
  }

  function handleTreeKeys(e) {
    const current = e.target.closest?.('.treeview-node-content');
    if (!current) return;
    const li = current.closest('li[data-node-data]'), node = JSON.parse(li.dataset.nodeData);
    const items = [...panel.querySelectorAll('#solveit-file-tree li[data-node-data] > .treeview-node-content')].filter(item => {
      const itemLi = item.closest('li[data-node-data]');
      if (itemLi.closest('li.hidden')) return false;
      let parent = itemLi.parentElement.closest('li[data-node-data]');
      while (parent) { if (!parent.classList.contains('expanded')) return false; parent = parent.parentElement.closest('li[data-node-data]'); }
      return true;
    });
    const index = items.indexOf(current);
    if (e.key === 'ArrowDown' || e.key === 'ArrowUp') {
      e.preventDefault();
      items[Math.max(0, Math.min(items.length - 1, index + (e.key === 'ArrowDown' ? 1 : -1)))]?.focus();
    } else if (e.key === 'ArrowRight' && node.kind === 'directory') {
      e.preventDefault();
      if (li.classList.contains('expanded')) li.querySelector('ul li[data-node-data] > .treeview-node-content')?.focus();
      else li.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click();
    } else if (e.key === 'ArrowLeft') {
      e.preventDefault();
      if (node.kind === 'directory' && li.classList.contains('expanded')) li.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click();
      else li.parentElement.closest('li[data-node-data]')?.querySelector(':scope > .treeview-node-content')?.focus();
    } else if (e.key === 'Enter' || e.key === ' ') {
      e.preventDefault();
      if (node.kind === 'directory') li.querySelector(':scope > .treeview-node-content > .treeview-expander')?.click();
      else if (node.href) openNode(node);
    } else if (e.key === 'F2') {
      e.preventDefault(); renameNode(node);
    } else if (e.key === 'Delete' || e.key === 'Backspace') {
      e.preventDefault(); deleteNode(node);
    } else if ((e.metaKey || e.ctrlKey) && !e.altKey && e.key.toLowerCase() === 'c') {
      e.preventDefault(); copyNode(node);
    } else if ((e.metaKey || e.ctrlKey) && !e.altKey && e.key.toLowerCase() === 'v') {
      e.preventDefault();
      const clip = getClipboard();
      if (clip) pasteNodeInto(clip, node.kind === 'directory' ? node.path : (node.parentPath || '.'));
    } else if (e.key === 'ContextMenu' || (e.shiftKey && e.key === 'F10')) {
      e.preventDefault(); showMenu({target: current, currentTarget: current, clientX: 0, clientY: 0, preventDefault() {}, stopPropagation() {}});
    }
  }

  function closePanel() {
    if (!panel) return;
    closeMenu(); closeUploadPop(); dismissCreate(); clearDropState(); panel.classList.remove('open'); panel.inert = true; previousFocus?.focus();
    saveOpen(false);
  }

  const onDocPointerDown = e => {
    if (menu && !menu.contains(e.target)) closeMenu();
    if (uploadPop && !uploadPop.contains(e.target) && !e.target.closest?.('.solveit-tree-upload, .solveit-tree-row-upload')) closeUploadPop();
    if (createDraft && !createDraft.input.readOnly && !createDraft.form.contains(e.target)) dismissCreate();
  };
  const onWinKeyDown = e => {
    if (document.querySelector('#solveit-launcher.open')) return;
    if (e.target.closest?.('.solveit-tree-create')) return;
    if (e.key === 'Escape' && (closeUploadPop() || closeMenu())) { e.preventDefault(); e.stopImmediatePropagation(); return; }
    const active = panel?.classList.contains('open') && panel.contains(document.activeElement);
    const isTreeMod = (e.metaKey || e.ctrlKey) && !e.altKey && ['c', 'v'].includes(e.key.toLowerCase());
    if (active && !menu && !e.target.closest?.('.solveit-tree-filter') && (isTreeMod || ['ArrowDown', 'ArrowUp', 'ArrowLeft', 'ArrowRight', 'Enter', ' ', 'ContextMenu', 'F10', 'F2', 'Delete', 'Backspace'].includes(e.key))) {
      if (e.key === 'F10' && !e.shiftKey) return;
      e.stopImmediatePropagation();
      handleTreeKeys(e);
      return;
    }
  };
  const ac = new AbortController(), cap = {capture: true, signal: ac.signal};
  on(document, 'pointerdown', onDocPointerDown, cap);
  on(window, 'keydown', onWinKeyDown, cap);

  function disable() {
    clearInterval(refreshTimer); refreshTimer = null;
    closeMenu(); closeUploadPop(); dismissCreate(); clearDropState();
    ac.abort();
    if (dlgNameNode && dlgNameOrigParent) {
      if (dlgNameOrigNext && dlgNameOrigNext.parentNode === dlgNameOrigParent) dlgNameOrigParent.insertBefore(dlgNameNode, dlgNameOrigNext);
      else dlgNameOrigParent.append(dlgNameNode);
    }
    if (workspaceRow) {
      const scroller = workspaceRow.querySelector(':scope > .overflow-y-auto');
      if (scroller) workspaceRow.before(scroller);
      workspaceRow.remove();
      workspaceRow = null;
    }
    panel?.remove();
    panel = null; tree = null;
  }

  const offMobile = window.craftMobile?.onChange(() => { if (tree) renderTree(); });
  window.solveitFileTree = {
    toggle, open: openNode, close: closePanel,
    isOpen: () => !!panel?.classList.contains('open'),
    disable() { offMobile?.(); disable(); },
    async entries() {  // every loaded dialog/file, for the launcher
      if (!rowsByPath.has('.')) { rowsByPath.set('.', await getRows('.')); fetchedAtByPath.set('.', Date.now()); saveCache(); }
      return [...rowsByPath.entries()]
        .filter(([p]) => !hideDots || !p.split('/').some(part => part !== '.' && part.startsWith('.')))
        .flatMap(([, rows]) => rows)
        .filter(n => n.kind !== 'directory' && (!hideDots || !(n.filename || n.name).startsWith('.')));
    },
  };
  function restoreOpen() {
    if (window.craftMobile?.active()) return;  // a drawer popping open over the page on load is just in the way on a phone
    if (store.get(OPEN_KEY, null, sessionStorage) === '1' && !panel?.classList.contains('open')) toggle(false);
  }
  if (document.readyState === 'loading') on(document, 'DOMContentLoaded', restoreOpen, {once: true, signal: ac.signal});
  else restoreOpen();
})();

(() => {
  function markDotPrompts() {
    document.querySelectorAll('.editable').forEach(editable => {
      const prompt = editable.querySelector(
        '.msg-card.card-in[data-mtype="prompt"]'
      );
      const body = prompt?.querySelector('.msg-body');
      const isDot = !!body && ['', '.'].includes(body.textContent.trim()) && !body.querySelector('img,video,audio,iframe,canvas,svg');

      editable.classList.toggle('dot-prompt', isDot);
      const response = editable.querySelector(':scope > .card-out .msg-content');
      const hasResponse = !!response && (!!response.textContent.trim() || !!response.querySelector('img,video,audio,iframe,canvas,svg'));
      editable.classList.toggle('dot-prompt-response', isDot && hasResponse);
    });
  }

  markDotPrompts();
  window.__craftCMCore.onDomChange(markDotPrompts, {delay: 'raf', inDialog: true, charData: true});
})();

## File Tree CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Lightweight file tree prototype ─────────────────────────────────── */
.solveit-tree-workspace {
  display: flex;
  flex: 1 1 0%;
  min-width: 0;
  min-height: 0;
  overflow: hidden;
}
.solveit-tree-workspace > .overflow-y-auto { flex: 1 1 0%; min-width: 0; min-height: 0; }
#solveit-file-tree-panel {
  position: relative;
  z-index: 50;
  display: flex;
  flex-direction: column;
  flex: 0 0 0;
  width: 0;
  min-width: 0;
  max-width: 90vw;
  height: 100%;
  padding: 0;
  box-sizing: border-box;
  overflow: hidden;
  visibility: hidden;
  pointer-events: none;
  background: #1a1b26;
  color: #cfc9c2;
  box-shadow: none;
  transition: flex-basis .16s ease, padding .16s ease;
}
#solveit-file-tree-panel.open {
  flex: 0 0 auto;
  width: var(--solveit-tree-w, min(320px, 90vw));
  min-width: 200px;
  padding: 10px 10px 0 8px;
  visibility: visible;
  pointer-events: auto;
  border-right: 1px solid #ffffff10;
  box-shadow: 8px 0 24px #0008;
}
#solveit-file-tree-panel.resizing { transition: none; user-select: none; }
#solveit-file-tree-panel > :not(#solveit-file-tree):not(.solveit-tree-resizer) { flex: 0 0 auto; }
#solveit-file-tree-panel .solveit-tree-resizer {
  position: absolute; top: 0; right: -2px; bottom: 0; width: 6px; z-index: 60;
  cursor: col-resize; background: transparent; transition: background .12s ease;
}
#solveit-file-tree-panel .solveit-tree-resizer:hover,
#solveit-file-tree-panel.resizing .solveit-tree-resizer { background: #7aa2f788; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-resizer { display: none; }
#solveit-file-tree-panel > header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; }
#solveit-file-tree-panel .solveit-tree-brand { display: flex; align-items: baseline; gap: 6px; margin-bottom: 6px; min-width: 0; }
#solveit-file-tree-panel .solveit-tree-appname { flex: 0 0 auto; font-weight: 700; font-size: 13px; letter-spacing: .02em; color: #cfc9c2; }
#solveit-file-tree-panel h2#dlg-name { flex: 1 1 0%; min-width: 0; font-size: 12.5px !important; line-height: 1.3 !important; font-weight: 400 !important; padding: 0 !important; color: #cfc9c2cc !important; }
#solveit-file-tree-panel > header button { display: inline-flex; align-items: center; justify-content: center; border: 0; background: transparent; color: inherit; font-size: 1.25rem; line-height: 1; padding: 2px; cursor: pointer; }
#solveit-file-tree-panel .solveit-tree-shortcut { margin-left: auto; color: #cfc9c288; font: 11px monospace; }
#solveit-file-tree-panel .solveit-tree-refresh { margin-left: 6px; }
#solveit-file-tree-panel .treeview-node-content:focus-visible { outline: none; box-shadow: inset 0 0 0 2px #7aa2f7; background: #30364d; }
#solveit-file-tree-panel .solveit-tree-close { margin-left: 6px; }
#solveit-file-tree-panel .solveit-tree-back { margin-left: auto; }
#solveit-file-tree-panel .solveit-tree-back:disabled { opacity: .35; cursor: default; }
#solveit-file-tree-panel .solveit-tree-up { margin-left: 6px; }
.solveit-file-tree-path { color: #cfc9c299; font: 12px monospace; margin-bottom: 8px; overflow: hidden; text-overflow: ellipsis; }
#solveit-file-tree-panel .custom-treeview-wrapper { max-width: none; margin: 0; padding: 0; border: 0; border-radius: 0; background: transparent; box-shadow: none; }
#solveit-file-tree-panel .custom-treeview-wrapper > ul { padding-left: 0; }
#solveit-file-tree-panel .custom-treeview-wrapper ul ul {
  padding-left: 13px;
  margin-left: 7px;
  border-left: 1px solid #ffffff12;
}
#solveit-file-tree-panel .custom-treeview-wrapper li { margin: 2px 0; color: #cfc9c2; }
#solveit-file-tree-panel .custom-treeview-wrapper .treeview-node-content { padding: 4px; border-radius: 4px; }
#solveit-file-tree-panel .custom-treeview-wrapper .treeview-node-content:hover { background: #222434; }
#solveit-file-tree-panel .treeview-expander {
  display: inline-flex; align-items: center; justify-content: center;
  width: 14px; flex: 0 0 14px; margin-right: 2px; padding: 0;
  color: #787c99; font-size: 10px; line-height: 1;
  transition: transform .12s ease, color .12s ease;
}
#solveit-file-tree-panel .treeview-expander::before { content: "▸"; }
#solveit-file-tree-panel .treeview-expander > * { display: none; }
#solveit-file-tree-panel li.expanded > .treeview-node-content > .treeview-expander { transform: rotate(90deg); color: #a9b1d6; }
#solveit-file-tree-panel .treeview-expander-placeholder { width: 14px; flex: 0 0 14px; margin-right: 2px; padding: 0; }
#solveit-file-tree-panel .solveit-tree-new { margin-left: 6px; font-weight: 600; color: #7aa2f7; }
#solveit-file-tree-panel .solveit-tree-new:hover { color: #c0caf5; }
#solveit-file-tree-panel img.solveit-tree-icon { width: 16px !important; height: 16px !important; max-width: 16px !important; max-height: 16px !important; flex: 0 0 16px; object-fit: contain; margin-right: 7px; }

#solveit-file-tree-panel .solveit-tree-filter { box-sizing: border-box; width: 100%; margin: 8px 0; padding: 6px 8px; border: 1px solid #ffffff22; border-radius: 4px; background: #16161e; color: #cfc9c2; }
#solveit-file-tree-panel .solveit-tree-status { color: #cfc9c299; font-size: 12px; }
#solveit-temp-editor { position: fixed; inset: 0; z-index: 1000; display: none; flex-direction: column; background: #16161e; color: #cfc9c2; }
#solveit-temp-editor.open { display: flex; }
body:has(#solveit-temp-editor.open) #solveit-file-tree-panel.open {
  position: fixed;
  inset-block: 0;
  left: 0;
  height: auto;
  z-index: 1010;
}
#solveit-temp-editor .solveit-temp-editor-bar { display: flex; flex: 0 0 36px; align-items: center; justify-content: space-between; padding: 0 12px; background: #1a1b26; border-bottom: 1px solid #ffffff22; }
#solveit-temp-editor .solveit-temp-editor-bar button { border: 0; background: transparent; color: inherit; font-size: 1.4rem; cursor: pointer; }
#solveit-temp-editor iframe { flex: 1; width: 100%; min-height: 0; border: 0; background: #16161e; }

/* ── File tree: header toggles, context menu ───────────────────────── */
#solveit-file-tree-panel .solveit-tree-upload,
#solveit-file-tree-panel .solveit-tree-collapse { margin-left: 6px; color: #cfc9c2aa; }
#solveit-file-tree-panel .solveit-tree-upload:hover,
#solveit-file-tree-panel .solveit-tree-collapse:hover { color: #cfc9c2; }
#solveit-file-tree-panel .solveit-tree-dots { margin-left: 6px; font: 600 .95rem monospace; color: #cfc9c288; }
#solveit-file-tree-panel .solveit-tree-dots[aria-pressed="true"] { color: #7aa2f7; filter: drop-shadow(0 0 4px #7aa2f7aa); }
#solveit-file-tree-panel .solveit-tree-home { margin-left: 6px; }
.solveit-tree-menu { position: fixed; z-index: 1100; min-width: 170px; padding: 4px; background: #1f2030; border: 1px solid #ffffff1a; border-radius: 6px; box-shadow: 0 8px 24px #000a; }
.solveit-tree-menu button { display: block; width: 100%; text-align: left; padding: 5px 10px; border: 0; border-radius: 4px; background: transparent; color: #cfc9c2; font-size: 12px; cursor: pointer; }
.solveit-tree-menu button:hover:not(:disabled), .solveit-tree-menu button:focus-visible { background: #2b2e43; outline: none; }
.solveit-tree-menu button:disabled { color: #cfc9c255; cursor: default; }
.solveit-tree-menu button.danger { color: #f7768e; }
.solveit-tree-menu hr.solveit-tree-menu-sep { margin: 4px 2px; border: 0; border-top: 1px solid #ffffff16; }
#solveit-file-tree-panel .solveit-tree-rootbar { display: flex; align-items: center; gap: 6px; padding: 4px; margin-bottom: 4px; border-radius: 4px; font: 11.5px monospace; color: #cfc9c299; background: #16161e88; border: 1px solid transparent; user-select: none; cursor: default; }
#solveit-file-tree-panel .solveit-tree-rootbar:hover { background: #222434; color: #cfc9c2; }
#solveit-file-tree-panel .solveit-tree-rootbar-hint { margin-left: auto; font-size: 10px; color: #cfc9c255; }
#solveit-file-tree-panel .solveit-tree-row-upload { opacity: 0; pointer-events: none; flex: 0 0 auto; display: inline-flex; align-items: center; justify-content: center; width: 20px; height: 20px; margin-left: 4px; padding: 0; border: 0; border-radius: 4px; background: transparent; color: #cfc9c299; cursor: pointer; }
#solveit-file-tree-panel .treeview-node-content:is(:hover, :focus-within) .solveit-tree-row-upload,
#solveit-file-tree-panel .solveit-tree-rootbar:is(:hover, :focus-within) .solveit-tree-row-upload { opacity: 1; pointer-events: auto; }
#solveit-file-tree-panel .solveit-tree-row-upload:is(:hover, :focus-visible) { background: #30364d; color: #7aa2f7; outline: none; }
@media (hover: none) { #solveit-file-tree-panel .solveit-tree-row-upload { opacity: 1; pointer-events: auto; } }
.solveit-tree-upload-pop { position: fixed; z-index: 1100; width: 230px; padding: 8px; background: #1f2030; border: 1px solid #ffffff1a; border-radius: 6px; box-shadow: 0 8px 24px #000a; color: #cfc9c2; font-size: 12px; }
.solveit-tree-upload-head { display: flex; align-items: center; gap: 6px; margin-bottom: 6px; }
.solveit-tree-upload-dest { flex: 1 1 0%; min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; font: 11px monospace; color: #cfc9c2cc; }
.solveit-tree-upload-x { flex: 0 0 auto; border: 0; background: transparent; color: #cfc9c299; font-size: 15px; line-height: 1; cursor: pointer; padding: 0 2px; }
.solveit-tree-upload-x:hover { color: #cfc9c2; }
.solveit-tree-upload-drop { padding: 12px 8px; margin-bottom: 8px; border: 1px dashed #ffffff2e; border-radius: 4px; background: #16161e; color: #cfc9c288; text-align: center; font-size: 11.5px; }
.solveit-tree-upload-drop.dragover { border-color: #7aa2f7; background: #7aa2f71c; color: #cfc9c2; }
.solveit-tree-upload-actions { display: flex; gap: 6px; }
.solveit-tree-upload-actions button { flex: 1 1 0%; padding: 5px 8px; border: 1px solid #ffffff1f; border-radius: 4px; background: #242638; color: #cfc9c2; font-size: 12px; cursor: pointer; }
.solveit-tree-upload-actions button:is(:hover, :focus-visible) { background: #30364d; border-color: #7aa2f7; outline: none; }
#solveit-file-tree { flex: 1 1 0%; min-height: 0; overflow-y: auto; overscroll-behavior: contain; padding-bottom: 40px; }
#solveit-file-tree .treeview-node-content.solveit-tree-current { background: #7aa2f71c; box-shadow: inset 2px 0 0 #7aa2f7; }
#solveit-file-tree li.solveit-tree-dragging { opacity: .4; }
#solveit-file-tree-panel .solveit-tree-drop-target { background: #7aa2f728 !important; box-shadow: inset 0 0 0 1.5px #7aa2f7 !important; color: #cfc9c2 !important; }
#solveit-file-tree .solveit-tree-create { margin: 3px 4px 5px; }
#solveit-file-tree li > .solveit-tree-create { margin-left: 14px; }
#solveit-file-tree .solveit-tree-create input { box-sizing: border-box; width: 100%; min-width: 0; padding: 4px 6px; border: 1px solid #7aa2f7; border-radius: 4px; background: #16161e; color: #cfc9c2; font: inherit; }
#solveit-file-tree .solveit-tree-create input:focus-visible { outline: 1px solid #7aa2f7; outline-offset: 1px; }
#solveit-file-tree .solveit-tree-create input[aria-invalid="true"] { border-color: #f7768e; }
#solveit-tree-create-error:not(:empty) { margin-top: 4px; color: #f7768e; font-size: 11px; overflow-wrap: anywhere; }

## Live `fastcdp` Verification

In [ ]:
#| live
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    async with Rungs(page)('file tree open, populate, close'):
        await page.press('b', mod=True)
        await page.wait_for_selector('#solveit-file-tree-panel.open', timeout=3)
        await page.wait_for("document.querySelectorAll('#solveit-file-tree-panel.open [role=\"treeitem\"], #solveit-file-tree-panel.open li').length >= 1", timeout=5)
        await page.press('b', mod=True)
        await page.wait_for_selector('#solveit-file-tree-panel.open', present=False, timeout=3)
    assert not page_errors, f"Uncaught page errors: {page_errors}"